# MethylLlama — Quickstart

This notebook loads the pretrained MethylLlama checkpoint from HuggingFace,
runs it on the 120-sample demo dataset, and shows age predictions.

**What you need**
- Python 3.10+, PyTorch, and the packages in `requirements.txt`
- GPU recommended but not required (CPU runs slowly on 21k CpGs)

**Architecture reminder**
```
Input: N CpG sites (β-values + site IDs)
  → MethylLlamaEmbeddings  (CpG-ID embed + ScaleAdaptEncoder)
  → 4 × LlamaLayer         (RMSNorm + MHA/RoPE + SwiGLU, Pre-LN)
  → CLS pooler_output      (Linear + Tanh)
  → age head               (256D → 1)
```
The pretraining task was WCED (Whole-Cell Expression Decoder): the CLS token
learns to reconstruct all CpG β-values from a 50 % random input subset,
forcing it to summarise the full methylation profile.

In [ ]:
# Install / upgrade if running for the first time
# !pip install -q huggingface_hub anndata scanpy matplotlib

In [ ]:
import sys, os
from pathlib import Path

# Make sure bmfm_methylation is importable (works when notebook is in tutorials/)
repo_root = Path(".").resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Download checkpoint from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="netanelazran1/MethylLlama",
    filename="pretrain_llama_epoch98_val0.0059.ckpt",
)
print(f"Checkpoint at: {ckpt_path}")

## 2. Inspect checkpoint & build model config

In [ ]:
# Patch torch.load for PyTorch >= 2.6 (weights_only=True breaks Lightning ckpts)
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _orig_load(*args, **kwargs)
torch.load = _safe_load

ckpt = torch.load(ckpt_path, map_location="cpu")

# Determine vocab_size from the actual embedding table stored in the checkpoint
embed_weight = ckpt["state_dict"]["encoder.embeddings.cpg_sites_embeddings.weight"]
vocab_size   = embed_weight.shape[0]   # e.g. 49161 = 49156 CpGs + 5 special tokens
hidden_size  = embed_weight.shape[1]   # 256 for the 'small' architecture

print(f"Embedding table:  {vocab_size} × {hidden_size}")
print(f"Decoder vocab:    {ckpt['hyper_parameters']['vocab_size']} CpGs per batch (WCED subset_k)")
print(f"Hparams:          {ckpt['hyper_parameters']}")

In [ ]:
from bmfm_methylation.llama.model import MethylLlamaConfig
from bmfm_methylation.llama.wced_llama import WCEDLlamaModule

# Architecture: 'small' variant (256D × 4L × 4H), matches pretrain job 44450919
model_config = MethylLlamaConfig(
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=320,
    vocab_size=vocab_size,      # from checkpoint embedding table
    n_sin_basis=48,
    basis_scale=2.0,
)

decoder_vocab = ckpt["hyper_parameters"]["vocab_size"]
module = WCEDLlamaModule(
    model_config=model_config,
    vocab_size=decoder_vocab,
)
module.load_state_dict(ckpt["state_dict"])
module = module.to(DEVICE).eval()

encoder = module.encoder
n_params = sum(p.numel() for p in encoder.parameters())
print(f"Encoder loaded: {n_params/1e6:.1f}M parameters")

## 3. Load demo dataset

120 samples × 21,368 CpGs — stratified by age bin, diverse tissues.

In [ ]:
demo_path = Path("data/methylllama_demo_120samples.h5ad")
adata = ad.read_h5ad(demo_path)

print(f"Shape:    {adata.shape}")
print(f"Obs cols: {list(adata.obs.columns)}")
print(f"\nAge range: {float(adata.obs['age'].min()):.0f} – {float(adata.obs['age'].max()):.0f} yr")
print(f"Tissues (top 5):\n{adata.obs['tissue_type'].value_counts().head()}")

## 4. Tokenize

MethylLlama expects dual-field input: `[B, 2, L]`
- Field 0: CpG-site token IDs (integer)
- Field 1: β-values (float, −2.0 marks the CLS position)

We build a lightweight tokenizer from the demo CpG site names.
The CpG-ID embeddings use indices within the pretrained table, so the
beta-value encoder (ScaleAdaptEncoder) is fully correct while the CpG-identity
embedding is approximate — good enough for a demo.

In [ ]:
from transformers import BertTokenizerFast
import tempfile, json

def build_demo_tokenizer(cpg_names):
    """Build a BertTokenizerFast from the demo CpG site names."""
    special = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
    vocab_lines = special + list(cpg_names)

    with tempfile.TemporaryDirectory() as tmp:
        vocab_file = os.path.join(tmp, "vocab.txt")
        with open(vocab_file, "w") as f:
            f.write("\n".join(vocab_lines))

        tok = BertTokenizerFast(
            vocab_file=vocab_file,
            do_lower_case=False,
            tokenize_chinese_chars=False,
        )
    return tok


cpg_names = list(adata.var_names)
tok = build_demo_tokenizer(cpg_names)

cls_id  = tok.cls_token_id   # 2
unk_id  = tok.unk_token_id   # 1
pad_id  = tok.pad_token_id   # 0
print(f"Demo tokenizer: {len(tok)} tokens  |  CLS={cls_id}  UNK={unk_id}")

In [ ]:
def tokenize_adata(adata, tokenizer, n_cpg=4000, seed=42):
    """
    Convert an h5ad into MethylLlama inputs.

    Selects n_cpg sites at random per call (matches WCED pretraining input ratio).
    Returns:
        input_ids:      [N, 2, n_cpg+1]  (field 0 = cpg_ids, field 1 = beta_values)
        attention_mask: [N, n_cpg+1]
        ages:           [N] float tensor
    """
    rng = np.random.default_rng(seed)

    X = adata.X
    if hasattr(X, "toarray"):
        X = X.toarray()
    X = X.astype(np.float32)

    n_samples, n_total = X.shape
    n_cpg = min(n_cpg, n_total)

    # Random subset of CpG sites (consistent across all samples in this call)
    sel_idx = rng.choice(n_total, size=n_cpg, replace=False)
    sel_cpg_names = [cpg_names[i] for i in sel_idx]
    sel_token_ids = tokenizer.convert_tokens_to_ids(sel_cpg_names)  # list of ints

    L = n_cpg + 1  # +1 for CLS

    all_input_ids = []
    all_masks     = []

    for i in range(n_samples):
        betas = X[i, sel_idx].copy()   # [n_cpg]
        # Impute NaN with 0.5 (neutral methylation)
        betas[np.isnan(betas)] = 0.5

        # Field 0: CpG-site token IDs  [CLS_id, cpg1_id, cpg2_id, ...]
        cpg_ids   = torch.tensor([cls_id] + sel_token_ids, dtype=torch.float32)
        # Field 1: β-values  [-2.0 marks CLS special token, then real values]
        beta_vals = torch.tensor([-2.0] + betas.tolist(), dtype=torch.float32)

        input_ids = torch.stack([cpg_ids, beta_vals])  # [2, L]
        mask      = torch.ones(L, dtype=torch.long)

        all_input_ids.append(input_ids)
        all_masks.append(mask)

    input_ids_t      = torch.stack(all_input_ids)  # [N, 2, L]
    attention_mask_t = torch.stack(all_masks)        # [N, L]
    ages_t = torch.tensor(adata.obs["age"].astype(float).values, dtype=torch.float32)

    return input_ids_t, attention_mask_t, ages_t


input_ids, attention_mask, ages = tokenize_adata(adata, tok, n_cpg=4000)
print(f"input_ids:      {tuple(input_ids.shape)}")
print(f"attention_mask: {tuple(attention_mask.shape)}")
print(f"ages:           {tuple(ages.shape)}  min={ages.min():.0f}  max={ages.max():.0f}")

## 5. Extract CLS embeddings

`pooler_output` = Linear(CLS hidden state) + Tanh → global methylation summary

In [ ]:
BATCH_SIZE = 16
all_cls = []

with torch.no_grad():
    for start in range(0, len(input_ids), BATCH_SIZE):
        ids_b  = input_ids[start:start+BATCH_SIZE].to(DEVICE)
        mask_b = attention_mask[start:start+BATCH_SIZE].to(DEVICE)

        out = encoder(input_ids=ids_b, attention_mask=mask_b)
        all_cls.append(out.pooler_output.cpu())

cls_embeddings = torch.cat(all_cls, dim=0).numpy()  # [120, 256]
print(f"CLS embeddings: {cls_embeddings.shape}")
print(f"Norm (mean ± std): {np.linalg.norm(cls_embeddings, axis=1).mean():.3f} "
      f"± {np.linalg.norm(cls_embeddings, axis=1).std():.3f}")

## 6. Age predictions from the pretrain age head

The WCED pretraining objective includes an auxiliary age regression head
to prevent CLS collapse. Even without fine-tuning, the pretrained age head
gives a rough age estimate.

In [ ]:
cls_t = torch.tensor(cls_embeddings)
with torch.no_grad():
    pred_ages = module.age_head(cls_t.to(DEVICE)).squeeze(-1).cpu().numpy()

true_ages = ages.numpy()

# Pearson correlation
from scipy.stats import pearsonr
r, pval = pearsonr(true_ages, pred_ages)
print(f"Pretrain age head  |  Pearson r = {r:.3f}  (p = {pval:.2e})")
print("(Fine-tuned model achieves R² = 0.905, MedAE = 3.65 yr)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(true_ages, pred_ages, c=true_ages, cmap="viridis", s=50, alpha=0.8)
plt.colorbar(sc, ax=ax, label="True age (yr)")

lim = [min(true_ages.min(), pred_ages.min()) - 5,
       max(true_ages.max(), pred_ages.max()) + 5]
ax.plot(lim, lim, "k--", lw=1, label="Perfect prediction")

ax.set_xlabel("True age (yr)", fontsize=12)
ax.set_ylabel("Predicted age (yr)", fontsize=12)
ax.set_title(f"MethylLlama pretrain age head\nPearson r = {r:.3f}", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig("pretrain_age_scatter.png", dpi=150)
plt.show()
print("Saved: pretrain_age_scatter.png")

## Next steps

- **Embedding analysis**: see `embedding_analysis.ipynb` for UMAP visualisation of the CLS embeddings coloured by age and tissue type
- **Fine-tuning**: see `age_prediction.ipynb` for the full fine-tuning pipeline and paper results
- **Cluster training**: scripts in `scripts/llama/` and `scripts/wced/` for SLURM submission